In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/playground-series-s6e6/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e6/train.csv
/kaggle/input/competitions/playground-series-s6e6/test.csv


In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, accuracy_score
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

# =========================================================================
# 1. SETUP RAW DATA SPLITS 
# =========================================================================
train_df = pd.read_csv('/kaggle/input/competitions/playground-series-s6e6/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/playground-series-s6e6/test.csv')

# Separate independent target vectors
x_raw = train_df.drop(columns=['id', 'class'])
y = train_df['class']
target_map = {'GALAXY': 0, 'QSO': 1, 'STAR': 2}
y_encoded = y.map(target_map)

# Extract independent test features
x_test_raw = test_df.drop(columns=['id'])

# =========================================================================
# 2. FEATURE ENGINEERING (Applied to both Train and Test raw sets)
# =========================================================================
def engineer_stellar_features(df):
    df_out = df.copy()
    
    # 1. Standard Color Indices
    df_out['u_g_index'] = df_out['u'] - df_out['g']
    df_out['g_r_index'] = df_out['g'] - df_out['r']
    df_out['r_i_index'] = df_out['r'] - df_out['i']
    df_out['i_z_index'] = df_out['i'] - df_out['z']
    df_out['u_r_index'] = df_out['u'] - df_out['r']

    # 2. Total Brightness Proxy
    df_out['total_brightness_proxy'] = -2.5 * np.log10(
        10**(-0.4 * df_out['u']) + 10**(-0.4 * df_out['g']) + 
        10**(-0.4 * df_out['r']) + 10**(-0.4 * df_out['i']) + 
        10**(-0.4 * df_out['z'])
    )

    # 3. Redshift Interactions
    df_out['u_g_redshift'] = df_out['u_g_index'] * df_out['redshift']
    df_out['g_r_redshift'] = df_out['g_r_index'] * df_out['redshift']

    # 4. Spectral Curvatures
    df_out['spectral_curvature_1'] = df_out['u_g_index'] - df_out['g_r_index']
    df_out['spectral_curvature_2'] = df_out['g_r_index'] - df_out['r_i_index']
    
    return df_out

x_engineered = engineer_stellar_features(x_raw)
x_test_engineered = engineer_stellar_features(x_test_raw)

# =========================================================================
# 3. UNIFIED ONE-HOT ENCODING (Aligns Columns Between Train and Test Perfectly)
# =========================================================================
combined = pd.concat([x_engineered, x_test_engineered], keys=['train', 'test'])
cat_col = ['spectral_type', 'galaxy_population']
combined_encoded = pd.get_dummies(combined, columns=cat_col, dtype=int)

x_encoded = combined_encoded.xs('train')
x_test_encoded = combined_encoded.xs('test')

num_cols = [
    'alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift', 
    'u_g_index', 'g_r_index', 'r_i_index', 'i_z_index', 'u_r_index',
    'total_brightness_proxy', 'u_g_redshift', 'g_r_redshift',
    'spectral_curvature_1', 'spectral_curvature_2'
]

# =========================================================================
# 4. PYTORCH COMPONENTS FOR THE NEURAL NETWORK TRACK (TRACK B)
# =========================================================================
class TabularDataset(Dataset):
    def __init__(self, X, y=None):
        X_val = X.values if isinstance(X, pd.DataFrame) else X
        self.X = torch.tensor(X_val, dtype=torch.float32)
        self.y = torch.tensor(y.values, dtype=torch.long) if y is not None else None

    def __len__(self): return len(self.X)
    def __getitem__(self, idx):
        if self.y is not None: return self.X[idx], self.y[idx]
        return self.X[idx]

class RealMLP(nn.Module):
    def __init__(self, input_dim, output_dim=3, hidden_dims=[256, 128], dropout=0.2):
        super().__init__()
        layers = []
        in_dim = input_dim
        for h_dim in hidden_dims:
            layers.append(nn.Linear(in_dim, h_dim))
            layers.append(nn.LayerNorm(h_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            in_dim = h_dim
        self.mlp = nn.Sequential(*layers)
        self.out = nn.Linear(in_dim, output_dim)
        
    def forward(self, x): return self.out(self.mlp(x))

# Optimized batch_size from 30 to 512 for accelerated training performance
def train_predict_nn_fold(X_train, y_train, X_val, X_test, epochs=30, batch_size=512, lr=1e-3):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    input_dim = X_train.shape[1]
    
    train_loader = DataLoader(TabularDataset(X_train, y_train), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(TabularDataset(X_val), batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(TabularDataset(X_test), batch_size=batch_size, shuffle=False)
    
    model = RealMLP(input_dim=input_dim).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    
    model.train()
    for epoch in range(epochs):
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(batch_X), batch_y)
            loss.backward()
            optimizer.step()
            
    model.eval()
    val_preds, test_preds = [], []
    with torch.no_grad():
        for batch_X in val_loader:
            val_preds.append(torch.softmax(model(batch_X.to(device)), dim=1).cpu().numpy())
        for batch_X in test_loader:
            test_preds.append(torch.softmax(model(batch_X.to(device)), dim=1).cpu().numpy())
            
    return np.vstack(val_preds), np.vstack(test_preds)

# =========================================================================
# 5. KAGGLE CLOUD GPU-ACCELERATED HYPERPARAMETERS
# =========================================================================
lgb_params = {
    'objective': 'multiclass', 'num_class': 3, 'metric': 'multi_logloss',
    'boosting_type': 'gbdt', 'learning_rate': 0.05, 'num_leaves': 31,
    'max_depth': -1, 'feature_fraction': 0.8, 'random_state': 42, 'verbose': -1,
    'device': 'gpu' # Routes LightGBM matrix work to the cloud GPU
}

xgb_params = {
    'objective': 'multi:softprob', 'num_class': 3, 'eval_metric': 'mlogloss',
    'learning_rate': 0.05, 'max_depth': 6, 'subsample': 0.8,
    'colsample_bytree': 0.8, 'random_state': 42, 
    'tree_method': 'hist',
    'device': 'cuda' # Routes XGBoost structures straight into CUDA VRAM
}

cat_params = {
    'loss_function': 'MultiClass', 'eval_metric': 'MultiClass',
    'iterations': 500, # Safely bumped up for GPU runtimes
    'learning_rate': 0.05, 'depth': 6,
    'random_seed': 42, 'verbose': False,
    'task_type': 'GPU' # Runs CatBoost on Kaggle's GPU instance
}

# =========================================================================
# 6. MASTER CROSS-VALIDATION LOOP ENGINE
# =========================================================================
N_SPLITS = 5
N_CLASSES = 3
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

oof_lgb = np.zeros((len(x_encoded), N_CLASSES))
oof_xgb = np.zeros((len(x_encoded), N_CLASSES))
oof_cat = np.zeros((len(x_encoded), N_CLASSES))
oof_nn  = np.zeros((len(x_encoded), N_CLASSES))

test_lgb = np.zeros((len(x_test_encoded), N_CLASSES))
test_xgb = np.zeros((len(x_test_encoded), N_CLASSES))
test_cat = np.zeros((len(x_test_encoded), N_CLASSES))
test_nn  = np.zeros((len(x_test_encoded), N_CLASSES))

print("Commencing GPU-Accelerated Level-0 Cross-Validation Loops...\n")

for fold, (train_idx, val_idx) in enumerate(skf.split(x_encoded, y_encoded)):
    print(f"--- Processing Fold {fold + 1} / {N_SPLITS} ---")
    
    # TRACK A: Extract and Train Tree-Based Models (Raw Space)
    X_tr_trees, y_tr = x_encoded.iloc[train_idx], y_encoded.iloc[train_idx]
    X_val_trees, y_val = x_encoded.iloc[val_idx], y_encoded.iloc[val_idx]
    
    # 1. LightGBM Execution
    train_data_lgb = lgb.Dataset(X_tr_trees, label=y_tr)
    val_data_lgb = lgb.Dataset(X_val_trees, label=y_val, reference=train_data_lgb)
    model_lgb = lgb.train(lgb_params, train_data_lgb, num_boost_round=500, 
                          valid_sets=[val_data_lgb], callbacks=[lgb.early_stopping(50, verbose=False)])
    
    oof_lgb[val_idx] = model_lgb.predict(X_val_trees)
    test_lgb += model_lgb.predict(x_test_encoded) / N_SPLITS
    
    # 2. XGBoost Execution
    model_xgb = xgb.XGBClassifier(**xgb_params, n_estimators=500, early_stopping_rounds=50)
    model_xgb.fit(X_tr_trees, y_tr, eval_set=[(X_val_trees, y_val)], verbose=False)
    
    oof_xgb[val_idx] = model_xgb.predict_proba(X_val_trees)
    test_xgb += model_xgb.predict_proba(x_test_encoded) / N_SPLITS
    
    # 3. CatBoost Execution
    model_cat = CatBoostClassifier(**cat_params)
    model_cat.fit(X_tr_trees, y_tr, eval_set=(X_val_trees, y_val), early_stopping_rounds=50)
    
    oof_cat[val_idx] = model_cat.predict_proba(X_val_trees)
    test_cat += model_cat.predict_proba(x_test_encoded) / N_SPLITS
    
    # TRACK B: Extract and Train Deep Learning Models (Standardized Space)
    scaler = StandardScaler()
    X_tr_nn = X_tr_trees.copy()
    X_val_nn = X_val_trees.copy()
    X_test_nn = x_test_encoded.copy()
    
    X_tr_nn[num_cols] = scaler.fit_transform(X_tr_trees[num_cols])
    X_val_nn[num_cols] = scaler.transform(X_val_trees[num_cols])
    X_test_nn[num_cols] = scaler.transform(x_test_encoded[num_cols])
    
    fold_nn_oof, fold_nn_test = train_predict_nn_fold(X_tr_nn, y_tr, X_val_nn, X_test_nn)
    oof_nn[val_idx] = fold_nn_oof
    test_nn += fold_nn_test / N_SPLITS
    
    print(f"Fold {fold + 1} processing complete.\n")

print("All folds evaluated successfully. Assembling Stacking Generalization Arrays.")

# =========================================================================
# 7. CONCATENATE OOF PREDICTIONS TO ASSEMBLE LEVEL-1 DATA
# =========================================================================
X_meta_train = np.hstack([oof_lgb, oof_xgb, oof_cat, oof_nn])
X_meta_test  = np.hstack([test_lgb, test_xgb, test_cat, test_nn])

print("Upgraded Level-1 Meta-Train Matrix Shape:", X_meta_train.shape)
print("Upgraded Level-1 Meta-Test Matrix Shape : ", X_meta_test.shape)

# =========================================================================
# 8. LEVEL-1 META-MODEL STACKER BLENDING & INFERENCE
# =========================================================================
print("\nFinal Layer: Training Regularized Meta-Model Stacker...")
meta_model = LogisticRegression(max_iter=1000, C=0.1, multi_class='multinomial', random_state=42)
meta_model.fit(X_meta_train, y_encoded)

# Diagnostics
meta_oof_probs = meta_model.predict_proba(X_meta_train)
print(f"Final Combined Meta-OOF Multi-LogLoss: {log_loss(y_encoded, meta_oof_probs):.5f}")

# Generate Final CSV File
final_test_preds = meta_model.predict(X_meta_test)
inverse_target_map = {0: 'GALAXY', 1: 'QSO', 2: 'STAR'}

submission_df = pd.DataFrame({
    'id': test_df['id'],
    'class': pd.Series(final_test_preds).map(inverse_target_map)
})
submission_df.to_csv('submission.csv', index=False)
print("Output saved to 'submission.csv'. Ready for leaderboard evaluation!")

Commencing GPU-Accelerated Level-0 Cross-Validation Loops...

--- Processing Fold 1 / 5 ---


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [03:08:54] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, wh

Fold 1 processing complete.

--- Processing Fold 2 / 5 ---
Fold 2 processing complete.

--- Processing Fold 3 / 5 ---
Fold 3 processing complete.

--- Processing Fold 4 / 5 ---
Fold 4 processing complete.

--- Processing Fold 5 / 5 ---
Fold 5 processing complete.

All folds evaluated successfully. Assembling Stacking Generalization Arrays.
Upgraded Level-1 Meta-Train Matrix Shape: (577347, 12)
Upgraded Level-1 Meta-Test Matrix Shape :  (247435, 12)

Final Layer: Training Regularized Meta-Model Stacker...


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Final Combined Meta-OOF Multi-LogLoss: 0.11285
Output saved to 'submission.csv'. Ready for leaderboard evaluation!


In [3]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, accuracy_score

# =========================================================================
# 8. LEVEL-1 META-MODEL TRAINING & EVALUATION
# =========================================================================
print("\n Final Layer: Training Meta-Model Stacker...")

# We use strong L2 regularization (C=0.1) to softly blend the probabilities
meta_model = LogisticRegression(
    max_iter=100, 
    C=0.1, 
    multi_class='multinomial', 
    solver='lbfgs', 
    random_state=42
)

# Fit the meta-model on the cross-validated out-of-fold predictions
meta_model.fit(X_meta_train, y_encoded)

# Evaluate our training set ensemble performance
meta_oof_probs = meta_model.predict_proba(X_meta_train)
meta_oof_preds = meta_model.predict(X_meta_train)

oof_loss = log_loss(y_encoded, meta_oof_probs)
oof_acc = accuracy_score(y_encoded, meta_oof_preds)

print(f"\n Master Ensemble OOF Multi-LogLoss: {oof_loss:.5f}")
print(f" Master Ensemble OOF Accuracy:     {oof_acc:.5f}")

# =========================================================================
# 9. INFERENCE & SUBMISSION GENERATION
# =========================================================================
print("\n Generating final test predictions...")

# Generate final blended predictions for the test dataset
final_test_preds = meta_model.predict(X_meta_test)

# Map the integer predictions back to their original string labels
inverse_target_map = {0: 'GALAXY', 1: 'QSO', 2: 'STAR'}
submission_labels = pd.Series(final_test_preds).map(inverse_target_map)

# Build the submission dataframe using the test set identifiers
submission_df = pd.DataFrame({
    'id': test_df['id'],
    'class': submission_labels
})

# Save to disk
submission_df.to_csv('submission.csv', index=False)
print(" Submission file successfully exported to 'submission.csv'!")


 Final Layer: Training Meta-Model Stacker...


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(



 Master Ensemble OOF Multi-LogLoss: 0.11285
 Master Ensemble OOF Accuracy:     0.96611

 Generating final test predictions...
 Submission file successfully exported to 'submission.csv'!
